# OpenAlex Bronze-to-Silver evidence

This notebook reads the same mounted Bronze and Silver directories used by the scheduled container. It is intentionally read-only: it does not call OpenAlex and it never accesses credentials.

## Evidence checklist

The cells below display batch metadata, raw Bronze schema, row counts, valid Silver samples, and rejected-row reasons. An empty zone is reported clearly so the notebook is still safe to open before the first scheduled batch.

In [ ]:
from pathlib import Path
import json

from pyspark.sql import SparkSession, functions as F

BRONZE = Path("/data/bronze")
SILVER = Path("/data/silver")
spark = (
    SparkSession.builder.master("local[*]")
    .appName("openalex-pipeline-evidence")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)


In [ ]:
# Batch metadata is the audit index for ingestion and processing state.
metadata_paths = sorted((BRONZE / "batch_metadata").glob("*.json"))
metadata_rows = []
for path in metadata_paths:
    item = json.loads(path.read_text(encoding="utf-8"))
    metadata_rows.append({
        "batch_id": item.get("batch_id"),
        "mode": item.get("mode"),
        "status": item.get("status"),
        "record_count": item.get("record_count", 0),
        "page_count": item.get("page_count", 0),
        "created_at": item.get("created_at"),
        "completed_at": item.get("completed_at"),
    })
if metadata_rows:
    batch_metadata_df = spark.createDataFrame(metadata_rows)
    print(f"Batch metadata rows: {batch_metadata_df.count()}")
    batch_metadata_df.printSchema()
    display(batch_metadata_df.orderBy("created_at"))
else:
    print("No batch metadata has been written yet.")


In [ ]:
raw_page_paths = sorted((BRONZE / "openalex").glob("*/page-*.json"))
if raw_page_paths:
    raw_pages_df = spark.read.json([str(path) for path in raw_page_paths])
    print(f"Raw Bronze pages: {len(raw_page_paths)}")
    raw_pages_df.printSchema()
    print(f"Raw result rows: {raw_pages_df.select(F.explode("results").alias("work")).count()}")
else:
    print("No raw Bronze pages have been written yet.")


In [ ]:
silver_root = SILVER / "works"
if silver_root.exists() and list(silver_root.glob("*")):
    silver_df = spark.read.parquet(str(silver_root))
    print(f"Silver valid rows: {silver_df.count()}")
    silver_df.printSchema()
    display(silver_df.orderBy("openalex_id").limit(10))
else:
    print("No Silver Parquet batches have been written yet.")


In [ ]:
rejected_paths = sorted((BRONZE / "rejected").glob("*/part-*.json"))
if rejected_paths:
    rejected_df = spark.read.json([str(path) for path in rejected_paths])
    print(f"Rejected rows: {rejected_df.count()}")
    rejected_df.printSchema()
    display(
        rejected_df.groupBy("rejection_reason")
        .count()
        .orderBy(F.desc("count"))
    )
else:
    print("No rejected rows have been written yet.")
